# merge(), join(), and concat()

In [2]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105],
    'customer_name': ['Aman', 'Riya', 'Kabir', 'Neha', 'Arjun'],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai']
})

orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'customer_id': [101, 102, 101, 104, 106, 103],
    'sales': [2500, 1800, 3200, 1500, 4000, 2700],
    'status': [
        'Completed', 'Cancelled', 'Completed',
        'Completed', 'Completed', 'Cancelled'
    ]
})

products = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'category': ['Electronics', 'Clothing', 'Books', 'Electronics'],
    'price': [2500, 800, 400, 3200]
})
customers

,customer_id,customer_name,city
0,101,Aman,Delhi
1,102,Riya,Mumbai
2,103,Kabir,Delhi
3,104,Neha,Pune
4,105,Arjun,Mumbai


In [3]:
products

,product_id,category,price
0,1,Electronics,2500
1,2,Clothing,800
2,3,Books,400
3,4,Electronics,3200


In [4]:
orders

,order_id,customer_id,sales,status
0,1001,101,2500,Completed
1,1002,102,1800,Cancelled
2,1003,101,3200,Completed
3,1004,104,1500,Completed
4,1005,106,4000,Completed
5,1006,103,2700,Cancelled


#### Q1. Basic merge

Combine customers and orders using an inner join on customer_id. Display the resulting DataFrame.

In [6]:
merged_df=pd.merge(customers,orders,
                  on='customer_id',
                  how='inner')
merged_df

,customer_id,customer_name,city,order_id,sales,status
0,101,Aman,Delhi,1001,2500,Completed
1,101,Aman,Delhi,1003,3200,Completed
2,102,Riya,Mumbai,1002,1800,Cancelled
3,103,Kabir,Delhi,1006,2700,Cancelled
4,104,Neha,Pune,1004,1500,Completed


#### Q2. Left merge

Merge customers with orders using a left join. Keep every customer, including those without orders.

In [8]:
merge_df=pd.merge(customers,orders,
               on='customer_id',
               how='left')
merged_df

,customer_id,customer_name,city,order_id,sales,status
0,101,Aman,Delhi,1001,2500,Completed
1,101,Aman,Delhi,1003,3200,Completed
2,102,Riya,Mumbai,1002,1800,Cancelled
3,103,Kabir,Delhi,1006,2700,Cancelled
4,104,Neha,Pune,1004,1500,Completed


#### Q3. Right merge

Merge customers and orders using a right join. Keep every order, including orders whose customer details are missing.

In [10]:
merge_df=pd.merge(customers,orders,
               on='customer_id',
               how='right')
merged_df

,customer_id,customer_name,city,order_id,sales,status
0,101,Aman,Delhi,1001,2500,Completed
1,101,Aman,Delhi,1003,3200,Completed
2,102,Riya,Mumbai,1002,1800,Cancelled
3,103,Kabir,Delhi,1006,2700,Cancelled
4,104,Neha,Pune,1004,1500,Completed


#### Q4. Outer merge

Perform an outer merge of customers and orders. Identify rows that exist in only one DataFrame.

In [12]:
merged_df=pd.merge(customers,orders,
               on='customer_id',
               how='outer',
               indicator=True)
merged_df

,customer_id,customer_name,city,order_id,sales,status,_merge
0,101,Aman,Delhi,1001.0,2500.0,Completed,both
1,101,Aman,Delhi,1003.0,3200.0,Completed,both
2,102,Riya,Mumbai,1002.0,1800.0,Cancelled,both
3,103,Kabir,Delhi,1006.0,2700.0,Cancelled,both
4,104,Neha,Pune,1004.0,1500.0,Completed,both
5,105,Arjun,Mumbai,NaN,NaN,NaN,left_only
6,106,NaN,NaN,1005.0,4000.0,Completed,right_only


In [13]:
customers_only = merged_df[merged_df['_merge'] == 'left_only']
display(customers_only)

,customer_id,customer_name,city,order_id,sales,status,_merge
5,105,Arjun,Mumbai,NaN,NaN,NaN,left_only


In [26]:
orders_only=merged_df[merged_df['_merge']=='right_only']
display(orders_only)

,customer_id,customer_name,city,order_id,sales,status,_merge
6,106,NaN,NaN,1005.0,4000.0,Completed,right_only


#### Q5. Merge indicator

Repeat Q4 using indicator=True. Interpret the _merge column values.

#### Q6. Add customer details

Create a DataFrame containing order_id, customer_name, city, sales, and status by merging the two datasets.

In [30]:
merged_df=pd.merge(customers,orders,
                  how='inner',
                  on='customer_id')[['order_id', 'customer_name', 'city', 'sales', 'status']]
merged_df

,order_id,customer_name,city,sales,status
0,1001,Aman,Delhi,2500,Completed
1,1003,Aman,Delhi,3200,Completed
2,1002,Riya,Mumbai,1800,Cancelled
3,1006,Kabir,Delhi,2700,Cancelled
4,1004,Neha,Pune,1500,Completed


#### Q7. Find unmatched orders

Identify orders whose customer_id does not exist in the customers DataFrame.

In [49]:
merged_df=pd.merge(customers,orders,
                  how='left',
                  on='customer_id',
                  indicator=True)
invalid=merged_df[merged_df['_merge']=='left_only']
invalid

,customer_id,customer_name,city,order_id,sales,status,_merge
5,105,Arjun,Mumbai,NaN,NaN,NaN,left_only


#### Q8. Find customers without orders

Return customers who have no matching order.

In [53]:
merged_df=pd.merge(customers,orders,
                  how='left',
                  on='customer_id',
                  indicator=True)
no_orders=merged_df[merged_df['_merge']=='left_only']

no_orders

,customer_id,customer_name,city,order_id,sales,status,_merge
5,105,Arjun,Mumbai,NaN,NaN,NaN,left_only


#### Q9. Concatenate rows

Create another DataFrame:

In [56]:
new_customers = pd.DataFrame({
    'customer_id': [106, 107],
    'customer_name': ['Meera', 'Dev'],
    'city': ['Jaipur', 'Delhi']
})

Use pd.concat() to append these customers to customers, resetting the index.

In [67]:
updated_df=pd.concat([customers,new_customers]).reset_index()
updated_df

,index,customer_id,customer_name,city
0,0,101,Aman,Delhi
1,1,102,Riya,Mumbai
2,2,103,Kabir,Delhi
3,3,104,Neha,Pune
4,4,105,Arjun,Mumbai
5,0,106,Meera,Jaipur
6,1,107,Dev,Delhi


#### Q10. Business analysis

Create a customer-level report containing:
- customer_name
- city
- order_count
- total_sales
Include customers with zero orders and sort by total_sales in descending order.

In [72]:
report=pd.merge(customers,orders,
               on='customer_id',
               how='inner')[['customer_name','city','order_id','sales']]
display(report)

,customer_name,city,order_id,sales
0,Aman,Delhi,1001,2500
1,Aman,Delhi,1003,3200
2,Riya,Mumbai,1002,1800
3,Kabir,Delhi,1006,2700
4,Neha,Pune,1004,1500


In [84]:
report['order_count']=report.groupby('customer_name')['order_id'].transform('count')
report['total_sales']=report.groupby('customer_name')['sales'].transform('sum')
display(report)



,customer_name,city,order_id,sales,order_count,total_sales
0,Aman,Delhi,1001,2500,2,5700
1,Aman,Delhi,1003,3200,2,5700
2,Riya,Mumbai,1002,1800,1,1800
3,Kabir,Delhi,1006,2700,1,2700
4,Neha,Pune,1004,1500,1,1500
